# Validate Gold Tables

Lightweight post-pipeline validation for analytical tables in the `serve` schema.

Runs after `operations_gold_transformation_pipeline` in the master ingestion workflow.


## Configuration


In [ ]:
CATALOG = "jm_databricks_learning_ws"
SERVE_SCHEMA = "serve"
SERVE = f"{CATALOG}.{SERVE_SCHEMA}"

# Maximum age for the latest timestamp before freshness checks fail.
FRESHNESS_MAX_AGE_DAYS = 7

# Candidate timestamp columns checked when no table-specific column is configured.
DEFAULT_FRESHNESS_COLUMNS = (
    "updated_at",
    "created_at",
    "_fivetran_synced",
    "ingestion_timestamp",
    "snapshot_date",
)

# Per-table validation rules. Table names are discovered from the serve schema;
# only tables listed here are required to exist and receive full validation.
GOLD_TABLE_RULES = {
    "supplier_summary": {
        "primary_keys": ["supplier_id"],
        "required_columns": [
            "supplier_id",
            "total_purchase_orders",
            "total_quantity",
        ],
        "freshness_column": None,
    },
    "customer_summary": {
        "primary_keys": ["customer_id"],
        "required_columns": [
            "customer_id",
            "total_sales_orders",
            "total_quantity",
        ],
        "freshness_column": None,
    },
    "inventory_summary": {
        "primary_keys": ["warehouse_id"],
        "required_columns": [
            "warehouse_id",
            "current_stock",
            "total_materials",
        ],
        "freshness_column": None,
    },
    "material_summary": {
        "primary_keys": ["material_id"],
        "required_columns": [
            "material_id",
            "material_name",
            "material_type",
            "purchased_quantity",
            "sold_quantity",
            "current_inventory",
        ],
        "freshness_column": None,
    },
    "business_kpi_summary": {
        "primary_keys": ["inventory_snapshot_date"],
        "required_columns": [
            "total_sales_orders",
            "total_purchase_orders",
            "inventory_on_hand",
            "active_suppliers",
            "active_customers",
            "inventory_snapshot_date",
        ],
        "freshness_column": "inventory_snapshot_date",
    },
    "sales_trend_monthly": {
        "primary_keys": ["month_start_date"],
        "required_columns": [
            "month_start_date",
            "year_month",
            "order_count",
            "total_quantity",
        ],
        "freshness_column": None,
    },
    "purchase_trend_monthly": {
        "primary_keys": ["month_start_date"],
        "required_columns": [
            "month_start_date",
            "year_month",
            "order_count",
            "total_quantity",
        ],
        "freshness_column": None,
    },
    "top_selling_materials": {
        "primary_keys": ["material_id"],
        "required_columns": [
            "sales_rank",
            "material_id",
            "material_name",
            "sold_quantity",
        ],
        "freshness_column": None,
    },
}


## Imports


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from typing import Any

from pyspark.sql import DataFrame
from pyspark.sql.functions import col, max as spark_max


## Validation helpers


In [ ]:
@dataclass(frozen=True)
class ValidationResult:
    name: str
    passed: bool
    details: str


def discover_gold_tables(catalog: str, schema: str) -> list[str]:
    """List table names in the gold schema."""
    rows = spark.sql(f"SHOW TABLES IN `{catalog}`.`{schema}`").collect()
    return sorted(row.tableName for row in rows)


def validate_table_exists(catalog: str, schema: str, table_name: str) -> ValidationResult:
    """Confirm a gold table is registered and readable."""
    full_name = f"{catalog}.{schema}.{table_name}"
    validation_name = f"{table_name}.exists"
    try:
        spark.table(full_name).limit(1).collect()
        return ValidationResult(
            name=validation_name,
            passed=True,
            details=f"Table {full_name} exists and is readable.",
        )
    except Exception as exc:
        return ValidationResult(
            name=validation_name,
            passed=False,
            details=f"Table {full_name} is missing or unreadable: {exc}",
        )


def validate_row_count(df: DataFrame, table_name: str, minimum_rows: int = 1) -> ValidationResult:
    """Ensure the table is not empty."""
    row_count = df.count()
    passed = row_count >= minimum_rows
    return ValidationResult(
        name=f"{table_name}.row_count",
        passed=passed,
        details=f"row_count={row_count:,} (minimum={minimum_rows:,})",
    )


def validate_duplicates(df: DataFrame, table_name: str, key_columns: list[str]) -> ValidationResult:
    """Ensure business keys are unique."""
    missing_keys = [column_name for column_name in key_columns if column_name not in df.columns]
    if missing_keys:
        return ValidationResult(
            name=f"{table_name}.duplicates",
            passed=False,
            details=f"Key column(s) missing from table: {', '.join(missing_keys)}",
        )

    total_rows = df.count()
    distinct_keys = df.select(*key_columns).distinct().count()
    duplicate_count = total_rows - distinct_keys
    passed = duplicate_count == 0
    return ValidationResult(
        name=f"{table_name}.duplicates",
        passed=passed,
        details=(
            f"keys={key_columns}; total_rows={total_rows:,}; "
            f"distinct_keys={distinct_keys:,}; duplicate_rows={duplicate_count:,}"
        ),
    )


def validate_required_columns(
    df: DataFrame,
    table_name: str,
    required_columns: list[str],
) -> ValidationResult:
    """Ensure required business columns are present and non-null."""
    issues: list[str] = []
    for column_name in required_columns:
        if column_name not in df.columns:
            issues.append(f"{column_name}: missing column")
            continue
        null_count = df.filter(col(column_name).isNull()).count()
        if null_count > 0:
            issues.append(f"{column_name}: null_count={null_count:,}")

    passed = not issues
    details = "; ".join(issues) if issues else f"required_columns={required_columns}"
    return ValidationResult(
        name=f"{table_name}.required_columns",
        passed=passed,
        details=details,
    )


def _resolve_freshness_column(
    df: DataFrame,
    configured_column: str | None,
    default_candidates: tuple[str, ...],
) -> tuple[str | None, str | None]:
    """
    Resolve which column to use for freshness checks.

    Returns (column_name, error_message). error_message is set when a configured
    column is missing from the table.
    """
    if configured_column:
        if configured_column in df.columns:
            return configured_column, None
        return None, f"Configured freshness column '{configured_column}' is not present."

    for candidate in default_candidates:
        if candidate in df.columns:
            return candidate, None
    return None, None


def validate_freshness(
    df: DataFrame,
    table_name: str,
    freshness_column: str | None = None,
    max_age_days: int = FRESHNESS_MAX_AGE_DAYS,
    default_candidates: tuple[str, ...] = DEFAULT_FRESHNESS_COLUMNS,
) -> ValidationResult:
    """Verify the latest timestamp is recent when a suitable column exists."""
    column_name, config_error = _resolve_freshness_column(
        df, freshness_column, default_candidates
    )
    if config_error:
        return ValidationResult(
            name=f"{table_name}.freshness",
            passed=False,
            details=config_error,
        )
    if column_name is None:
        return ValidationResult(
            name=f"{table_name}.freshness",
            passed=True,
            details="Skipped: no freshness timestamp column configured or present.",
        )

    latest_value = df.agg(spark_max(col(column_name)).alias("latest_value")).collect()[0]["latest_value"]
    if latest_value is None:
        return ValidationResult(
            name=f"{table_name}.freshness",
            passed=False,
            details=f"column={column_name}; latest_value is NULL",
        )

    if isinstance(latest_value, datetime):
        latest_dt = (
            latest_value.replace(tzinfo=timezone.utc)
            if latest_value.tzinfo is None
            else latest_value.astimezone(timezone.utc)
        )
    elif isinstance(latest_value, date):
        latest_dt = datetime.combine(latest_value, datetime.min.time(), tzinfo=timezone.utc)
    else:
        return ValidationResult(
            name=f"{table_name}.freshness",
            passed=False,
            details=f"column={column_name}; unsupported type={type(latest_value).__name__}",
        )

    cutoff = datetime.now(timezone.utc) - timedelta(days=max_age_days)
    passed = latest_dt >= cutoff
    return ValidationResult(
        name=f"{table_name}.freshness",
        passed=passed,
        details=(
            f"column={column_name}; latest_value={latest_value}; "
            f"max_age_days={max_age_days}; cutoff={cutoff.isoformat()}"
        ),
    )


def validate_gold_table(
    catalog: str,
    schema: str,
    table_name: str,
    rules: dict[str, Any],
) -> list[ValidationResult]:
    """Run all validations for a single gold table."""
    results = [validate_table_exists(catalog, schema, table_name)]
    if not results[-1].passed:
        return results

    df = spark.table(f"{catalog}.{schema}.{table_name}")
    results.append(validate_row_count(df, table_name))
    results.append(validate_duplicates(df, table_name, rules["primary_keys"]))
    results.append(validate_required_columns(df, table_name, rules["required_columns"]))
    results.append(
        validate_freshness(
            df,
            table_name,
            freshness_column=rules.get("freshness_column"),
        )
    )
    return results


## Run validations


In [ ]:
expected_tables = sorted(GOLD_TABLE_RULES.keys())
discovered_tables = discover_gold_tables(CATALOG, SERVE_SCHEMA)

validation_results: list[ValidationResult] = []

# Every configured gold table must exist in the serve schema.
for table_name in expected_tables:
    if table_name not in discovered_tables:
        validation_results.append(
            ValidationResult(
                name=f"{table_name}.exists",
                passed=False,
                details=(
                    f"Expected gold table {SERVE}.{table_name} was not found. "
                    f"Discovered tables: {discovered_tables}"
                ),
            )
        )

# Validate each expected table that is present.
for table_name in expected_tables:
    if table_name not in discovered_tables:
        continue
    validation_results.extend(
        validate_gold_table(CATALOG, SERVE_SCHEMA, table_name, GOLD_TABLE_RULES[table_name])
    )

# Surface unexpected tables without failing the job.
unexpected_tables = sorted(set(discovered_tables) - set(expected_tables))
if unexpected_tables:
    validation_results.append(
        ValidationResult(
            name="serve.unexpected_tables",
            passed=True,
            details=(
                "Informational only: tables present in serve but not configured in "
                f"GOLD_TABLE_RULES: {unexpected_tables}"
            ),
        )
    )


## Validation summary


In [ ]:
summary_rows = [
    {
        "validation": result.name,
        "status": "PASS" if result.passed else "FAIL",
        "details": result.details,
    }
    for result in validation_results
]

summary_df = spark.createDataFrame(summary_rows)
summary_df.show(truncate=False)

failed_results = [result for result in validation_results if not result.passed]

if failed_results:
    print("\nGold validation failed:\n")
    for result in failed_results:
        print(f"  [FAIL] {result.name}: {result.details}")
    raise Exception(
        f"Gold table validation failed with {len(failed_results)} failing check(s)."
    )

print("\nGold validation succeeded. All checks passed.\n")
for result in validation_results:
    print(f"  [PASS] {result.name}: {result.details}")

dbutils.notebook.exit("SUCCESS")
